<a href="https://colab.research.google.com/github/itsdev-ai/Transformer-Neural-Network/blob/main/Machine_Translation_using_hugging_face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.8/114.8 kB 13.7 MB/s eta 0:00:00


**Import**

In [ ]:
import pandas as pd
from contractions import fix
import re
import unicodedata
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM,Seq2SeqTrainer,Seq2SeqTrainingArguments

**Load Dataset**

In [ ]:
df=pd.read_csv(r'Dataset_English_Hindi.csv')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130476 entries, 0 to 130475
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   English  130474 non-null  object
 1   Hindi    130164 non-null  object
dtypes: object(2)
memory usage: 2.0+ MB


In [ ]:
print(df.isnull().sum())
print(df.duplicated().sum())

English      2
Hindi      312
dtype: int64
2788


In [ ]:
df=df.dropna()
df=df.drop_duplicates()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 127375 entries, 0 to 130475
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   English  127375 non-null  object
 1   Hindi    127375 non-null  object
dtypes: object(2)
memory usage: 2.9+ MB


In [ ]:
df=df.sample(n=10000,random_state=42).reset_index(drop=True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   English  10000 non-null  object
 1   Hindi    10000 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB


**Text Preprocessing**

In [ ]:
def preprocess_hi(sentence):
  sentence=sentence.strip()
  sentence=unicodedata.normalize('NFKC',sentence)
  sentence=sentence.replace('\u200c','').replace('\u200d','')
  sentence=re.sub(r'([?.!,|])',r' \1 ',sentence)
  sentence=sentence.strip()
  return sentence

In [ ]:
def preprocess_en(sentence):
  sentence=fix(sentence)
  sentence=sentence.lower().strip()
  sentence=re.sub(r' +',' ',sentence)
  sentence=re.sub(r'([?.!,])',r' \1',sentence)
  sentence=sentence.strip()
  return sentence

In [ ]:
df['English']=df['English'].apply(lambda x:preprocess_en(x))
df['Hindi']=df['Hindi'].apply(lambda x:preprocess_hi(x))

In [ ]:
df.head(10)

,English,Hindi
0,"in any case , the phone provided to cronje by...",चावल ने क्रोनिए को जो फोन दिया था वह कालरा के ...
1,"many scholars - and , of course , there are ...",हालांकि अनेक विद्वान इस मत का विरोध करते हैं ...
2,who are trying to expose stories like this,जो ऐसे कांडों का पर्दाफाश करने की कोशिश कर रहे...
3,"you co-opt almost everybody ,","आप हर चीज़ में भागिदार हैं ,"
4,hardinge had a miraculous escape but his bodyg...,हार्डिंग बाल-बाल बच गये किन्तु उनके अंगरक्षक म...
5,our role is to ensure that our knowledge about...,हमारी भूमिका यह सुनिश्चित करने की है कि हमारी ...
6,so seer by the establishment of the peto deswa...,इसलिए शंकराचार्य ने इन पीठो की स्थापना करके दे...
7,"in the hilly terrain , wheat , makai , kodo , ...",हिमालयी भाग में गेहूँ मकई कोदो आलू आदि का खाना...
8,help with health costs,आरोग्य खर्चे के लिए आर्थिक मदद .
9,was where do i begin .,कि मैं कहाँ से शुरू करूँ .


**Load Pretrained Model**

In [ ]:
model_name='facebook/nllb-200-distilled-600M'
model=AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer=AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.46GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

...

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

**Spilt Dataset**

In [ ]:
train_data,test_data=train_test_split(df,random_state=42,test_size=0.2)
print(train_data.shape,test_data.shape)

(8000, 2) (2000, 2)


**Dataset Creation**

In [ ]:
train_dataset=Dataset.from_pandas(train_data,preserve_index=False)
test_dataset=Dataset.from_pandas(test_data,preserve_index=False)

In [ ]:
print(train_dataset['English'][0:3])
print(train_dataset['Hindi'][0:3])

['however  , till the government ensures that its schemes work and the benefits reach the 33 ,000 families children will continue to die  .', 'and now we take this material , combine this', 'the following method of detection is effective :']
['लेकिन जब तक सरकार यह इंतजाम नहीं करती कि उसकी योजनाएं लगू हों और उनके फायदे 33 , 000 परिवारों तक फंचें  ,  बच्चे मौत की नींद सोते रहेंगे  .', 'और अब हम इस सामग्री को ले  ,', 'उन्हें ढुँढने का निम्नलिखित तरीका प्रभावशाली हो सकता है']


**Preprocess Function**

In [ ]:
def preprocess_function(examples):
  model_inputs=tokenizer(examples['English'],max_length=15,truncation=True,padding=True)
  labels=tokenizer(examples['Hindi'],max_length=15,truncation=True,padding=True)
  model_inputs['labels']=labels['input_ids']
  return model_inputs

In [ ]:
train_dataset=train_dataset.map(preprocess_function,batched=True)
test_dataset=test_dataset.map(preprocess_function,batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

**Set Pytorch Format**

In [ ]:
train_dataset.set_format(type='torch',columns=['input_ids','attention_mask','labels'])
test_dataset.set_format(type='torch',columns=['input_ids','attention_mask','labels'])

In [ ]:
train_dataset[0]

{'input_ids': tensor([256047,  82960,    146,   3191,    349,  52360,  10391,   7866,   1482,
           6629,   4055,  29694,  10527,    540,      2]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
 'labels': tensor([256047,  10157,   6855,   7488,  12209,   4228,  91464, 199779,   3600,
          46564,   1182,  53642,  27663,  19445,      2])}

In [ ]:
test_dataset[0]

{'input_ids': tensor([256047,    108,  16701,  26214,    146, 103829,   9980,   1044,    202,
           5076,   7506,    146,     16,    610,      2]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
 'labels': tensor([256047,   4827,  84256,   1035,   3061,  34147,  56085,  20244,  85847,
          54239,  10093,   1371,  31652,  48630,      2])}

In [ ]:
model.config.decoder_start_token_id=tokenizer.convert_tokens_to_ids('hin_Deva')
tokenizer.src_lang='eng_Latn'
tokenizer.tgt_lang='hin_Deva'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
results_dir='/content/drive/MyDrive/results'
model_dir='/content/drive/MyDrive/my_trans-model'

os.makedirs(results_dir,exist_ok=True)
os.makedirs(model_dir,exist_ok=True)

**Training Argumants**

In [ ]:
training_args=Seq2SeqTrainingArguments(
    output_dir='./result',
    eval_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    save_strategy='epoch',
    fp16=True,
    predict_with_generate=True
)

**Trainer Initialize**

In [ ]:
trainer=Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)
# trainer.train()

**Model and Tokenizer Save**

In [ ]:
trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir)

loaded_model=AutoModelForSeq2SeqLM.from_pretrained(model_dir)
loaded_tokenizer=AutoTokenizer.from_pretrained(model_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

**Use this cell to start training from scratch**

In [ ]:
# def translate_text(text):
#   tokenizer.src_lang='eng_Latn'
#   inputs=loaded_tokenizer(text,return_tensors='pt',max_length=128,truncation=True)
#   outputs=loaded_model.generate(**inputs,decoder_start_token_id=tokenizer.convert_tokens_to_ids('hin_Deva'),max_length=128,num_beams=4,early_stopping=True)
#   translation=loaded_tokenizer.decode(outputs[0],skip_special_tokens=True)
#   return translation

**Use this cell to resume training from the saved model**

In [ ]:
def translate_text(text):
  loaded_tokenizer.src_lang='eng_Latn'
  inputs=loaded_tokenizer(text,return_tensors='pt',max_length=128,truncation=True)
  outputs=loaded_model.generate(**inputs,forced_bos_token_id=loaded_tokenizer.convert_tokens_to_ids('hin_Deva'),max_length=128,num_beams=4,early_stopping=True)
  translation=loaded_tokenizer.decode(outputs[0],skip_special_tokens=True)
  return translation

In [ ]:
text_to_translation='I saw.,what were you doing there.'
translation_text=translate_text(text_to_translation)
print('Hindi Translation :',translation_text)

Hindi Translation : मैंने देखा, तुम वहाँ क्या कर रहे थे.


In [ ]:
text_to_translation='Despite having been warned repeatedly about the consequences.'
translation_text=translate_text(text_to_translation)
print('Hindi Translation :',translation_text)

Hindi Translation : इसके परिणामों के बारे में बार-बार चेतावनी दी गई है।


In [ ]:
text_to_translation='he continued to make decision that were likely to jeopardize his entire career.'
translation_text=translate_text(text_to_translation)
print('Hindi Translation :',translation_text)

Hindi Translation : उसने निर्णय लेना जारी रखा जो उसके पूरे करियर को खतरे में डाल सकता था।


In [ ]:
text_to_translation='jeopardize'
translation_text=translate_text(text_to_translation)
print('Hindi Translation :',translation_text)

Hindi Translation : जोखिम में डालना


In [ ]:
text_to_translation='perspicacious'
translation_text=translate_text(text_to_translation)
print('Hindi Translation :',translation_text)

Hindi Translation : समझदार


In [ ]:
text_to_translation='perfidious'
translation_text=translate_text(text_to_translation)
print('Hindi Translation :',translation_text)

Hindi Translation : धोखेबाज


In [ ]:
text_to_translation='The subtle implications of his ambiguous statement are difficult to decipher.'
translation_text=translate_text(text_to_translation)
print('Hindi Translation :',translation_text)

Hindi Translation : उनके अस्पष्ट बयान के सूक्ष्म प्रभावों को समझने में कठिनाई है।
